In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 1, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    compute_batch_size,
    detect_device,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data

In [2]:
DATA_SET = 'rand_C'

df_train = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_train_v2.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_val_v2.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_test_v2.parquet'))


OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

L4  |  VRAM: 24 GB  |  MAX_BATCH=32,768  |  dtype=torch.bfloat16


## Hyperparameters

In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=32,768  adaptive BATCH_SIZE=32,768  INIT_LR=0.002828  n_train=1,716,781  steps/epoch~52
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [5]:
from itertools import combinations

FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

# Generate ALL possible combinations of EXTRA_FEATURES (0 to 7)
feature_combos = []
base_name = '+'.join(FEATURES_3)
feature_combos.append((base_name, FEATURES_3.copy()))  # Base case

for r in range(1, len(EXTRA_FEATURES) + 1):
    for combo in combinations(EXTRA_FEATURES, r):
        feats = FEATURES_3 + list(combo)
        name = '+'.join(feats)
        feature_combos.append((name, feats))

# Calculate breakdown for all sizes
n_plus_0 = 1
n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))
n_plus_4 = len(list(combinations(EXTRA_FEATURES, 4)))
n_plus_5 = len(list(combinations(EXTRA_FEATURES, 5)))
n_plus_6 = len(list(combinations(EXTRA_FEATURES, 6)))
n_plus_7 = len(list(combinations(EXTRA_FEATURES, 7)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')
print(f'  3F + 4:     {n_plus_4}')
print(f'  3F + 5:     {n_plus_5}')
print(f'  3F + 6:     {n_plus_6}')
print(f'  3F + 7:     {n_plus_7}')

Total feature combinations: 128
  Base (3F):  1
  3F + 1:     7
  3F + 2:     21
  3F + 3:     35
  3F + 4:     35
  3F + 5:     21
  3F + 6:     7
  3F + 7:     1


## Pre-allocate on GPU

In [6]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.31 GB / 24 GB  |  Free: 23.3 GB
Train: 1,716,781  Val: 490,509  Test: 245,255  Features: 10


## Analytic benchmark

In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 64.9134  RMSE = 0.016269
Coefficients: a = -0.078535, b = -0.092385, c = -0.081882


## Train all feature combinations

In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 128 MODELS
  GPU: L4  |  Batch: 32,768  |  AMP: True  |  Max epochs: 100

  [  1/128] delta+T+spy_ret                SSE=61.0714  Gain=+5.9%  ep=100  19.2s  elapsed=0.3min
  [ 25/128] delta+T+spy_ret+gamma+vega     SSE=59.8757  Gain=+7.8%  ep=100  12.9s  elapsed=5.5min
  [ 50/128] delta+T+spy_ret+vix_mom_lag+gamma+vega SSE=47.2056  Gain=+27.3%  ep=100  13.0s  elapsed=10.9min
  [ 75/128] delta+T+spy_ret+vix_lag+vix_mom+gamma+theta SSE=40.0621  Gain=+38.3%  ep=100  12.8s  elapsed=16.3min
  [100/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta SSE=34.0104  Gain=+47.6%  ep=100  12.9s  elapsed=21.8min
  [125/128] delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+theta+vega+rho SSE=45.0407  Gain=+30.6%  ep=100  12.9s  elapsed=27.2min
  [128/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta+vega+rho SSE=37.2196  Gain=+42.7%  ep=100  13.3s  elapsed=27.9min

Done: 27.9 min for 128 models (avg 13.1s/model)


## Results summary

In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,64.913429,0.000265,0.016269,0.005912,0.001246,0.002637,0.068266,None,None,None
1,delta+T+spy_ret,61.071384,0.000249,0.015780,0.006703,-0.000381,0.003629,0.123412,13.9s,5.92%,None
2,delta+T+spy_ret+vix_lag,72.010597,0.000294,0.017135,0.009803,-0.000082,0.006616,-0.033604,13.0s,-10.93%,-17.91%
3,delta+T+spy_ret+vix_mom_lag,72.755112,0.000297,0.017224,0.009821,-0.000329,0.006654,-0.044290,13.1s,-12.08%,-1.03%
4,delta+T+spy_ret+vix_mom,73.685379,0.000300,0.017333,0.009936,0.000881,0.006676,-0.057643,12.9s,-13.51%,-1.28%
...,...,...,...,...,...,...,...,...,...,...,...
125,delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+thet...,45.040691,0.000184,0.013552,0.007568,0.000636,0.005019,0.353509,12.9s,30.61%,4.14%
126,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta+ve...,46.609982,0.000190,0.013786,0.007734,-0.001360,0.005175,0.330984,12.8s,28.20%,-3.48%
127,delta+T+spy_ret+vix_mom_lag+vix_mom+gamma+thet...,45.794056,0.000187,0.013665,0.007379,0.001498,0.004800,0.342695,13.2s,29.45%,1.75%
128,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,37.219646,0.000152,0.012319,0.006533,0.000836,0.004100,0.465768,13.3s,42.66%,18.72%


## Top 10 by Gain vs Hull-White

In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,34.0104,0.011776,47.61,12.9
1,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,9,36.6708,0.012228,43.51,12.9
2,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,10,37.2196,0.012319,42.66,13.3
3,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,9,38.0694,0.012459,41.35,13.3
4,delta+T+spy_ret+vix_mom_lag+vix_mom+gamma+theta,7,38.6872,0.012560,40.40,13.4
5,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,39.0110,0.012612,39.90,12.8
6,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,39.1606,0.012636,39.67,13.0
7,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta,7,40.0621,0.012781,38.28,12.7
8,delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+thet...,8,40.1194,0.012790,38.20,13.2
9,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta+rho,8,40.2268,0.012807,38.03,13.0


## Best per group (3F, +1, +2, +3)

In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): delta+T+spy_ret
    SSE=61.0714  RMSE=0.015780  Gain=5.92%

+1 (4F): delta+T+spy_ret+theta
    SSE=70.1041  RMSE=0.016907  Gain=-8.00%

+2 (5F): delta+T+spy_ret+vix_lag+vix_mom_lag
    SSE=48.2435  RMSE=0.014025  Gain=25.68%

+3 (6F): delta+T+spy_ret+vix_mom+gamma+theta
    SSE=44.3803  RMSE=0.013452  Gain=31.63%



## Summary statistics

In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 27.7 min (0.46 hr)
Models trained: 128
Best overall: delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta (Gain=47.61%)


## Cleanup

In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.47 GB / 24 GB
Total training time: 27.7 min for 128 models
